In [2]:
import os
import json
import re

In [10]:
PATH_AUDITS = './../datasets/acr-12_2025/audits.json'
PATH_ENTITIES = './../datasets/niort/entities.json'
PATH_USERS = './../datasets/production/users.json'

# PATH_AUDITS = './../datasets/acr-12_2025/audits.json'
# PATH_ENTITIES = './../datasets/acr-12_2025/entities.json'
# PATH_USERS = './../datasets/production/users.json'

In [11]:
# Load the PATH_AUDITS json into dict
entities = {}

with open(PATH_ENTITIES, 'r') as file:
    entities = json.load(file)

entity_list = []
for entity in entities:
    entity_list.append(entity['id'])
print(f'Loaded {len(entities)} entities from {PATH_ENTITIES}')


Loaded 6778 entities from ./../datasets/niort/entities.json


In [22]:
# Load the PATH_AUDITS json into dict
all_audits = {}
with open(PATH_AUDITS, 'r') as file:
    all_audits = json.load(file)

# get only audits before 2024-12-31
audits = [audit for audit in all_audits if audit['date'] < '2024-12-31']

print(f'Loaded {len(all_audits)} audits from {PATH_AUDITS}, filtered to {len(audits)} audits before 2024-12-31')

Loaded 24245 audits from ./../datasets/acr-12_2025/audits.json, filtered to 21676 audits before 2024-12-31


In [23]:
# aggregate audits by entity id
audits_by_entity = {}
for ai, audit in enumerate(audits):
    entity_id = audit.get('entityId')
    if entity_id and entity_id in entity_list:
        if entity_id not in audits_by_entity:
            audits_by_entity[entity_id] = []
        audits_by_entity[entity_id].append(audit)
    else:        
        print(f'Entity {entity_id} not in entity_list')

print(f'Aggregated audits by entity, found {len(audits_by_entity)} entities with audits')

Entity 1fc9779b-9f3f-43bb-b224-b40ef89268ef not in entity_list
Entity 1024f2e1-7b3f-403c-a299-6d1b0ed484c5 not in entity_list
Entity e7fcf6ed-5455-4966-949b-8a999237d548 not in entity_list
Entity 09086671-2b7a-41f0-b8da-4b93f69083b0 not in entity_list
Entity 40d659ee-8689-4519-9a91-24799c5b1667 not in entity_list
Entity f2170cea-2ec4-430b-bf1c-82c1a13c2554 not in entity_list
Entity 12960e39-19a3-4dcd-ae93-2dbd9a6c5824 not in entity_list
Entity 1655e532-b7fd-4dc2-b7eb-620bdd9dfbf3 not in entity_list
Entity a7c92dc0-32af-4f26-97aa-77ef5e095050 not in entity_list
Entity 3dfe8ad7-bb8f-4819-9805-b2a4c0f2ac13 not in entity_list
Entity f0458cbc-67a5-4130-a133-415ef0f5d7b5 not in entity_list
Entity 02b45e78-79ba-4e4c-8c73-19031ad6ef1c not in entity_list
Entity 9b89fe27-af5e-4211-9887-09ac9d5295cb not in entity_list
Entity 69d70463-c6b4-4017-aa85-970896fa03a3 not in entity_list
Entity f6a56b1f-d3be-42f9-ada4-ce80f85e1eda not in entity_list
Entity b9abb2c4-1e39-4259-bb77-46484fdbcb53 not in enti

In [24]:
# iterate over entities, find all items in audits for the entityId and fill the
#  audits_by_user dictionary so the first key level is user, 
#  second key level is entityId, for each entityId on second level store the relevant audit item count, 
#  on top level, user should store the total count of audits

audits_by_user = {}
for entity_id, entity_audits in audits_by_entity.items():
    for audit in entity_audits:
        user_id = audit.get('user')
        if user_id:
            if user_id not in audits_by_user:
                audits_by_user[user_id] = {}
            audits_by_user[user_id][entity_id] = audits_by_user[user_id].get(entity_id, 0) + 1
        audits_by_user[user_id]['total'] = audits_by_user[user_id].get('total', 0) + 1
print(f'Aggregated audits by user, found {len(audits_by_user)} users with audits')

Aggregated audits by user, found 8 users with audits


In [25]:
# load the PATH_USERS json into dict where userId is the key

users = {}
with open(PATH_USERS, 'r') as file:
    users = json.load(file)

# Transform users list into a dictionary with userId as the key
users = {user['id']: user for user in users}
print(f'Loaded {len(users)} users from {PATH_USERS}')

Loaded 19 users from ./../datasets/production/users.json


In [38]:
# For each user in audits_by_user, find the user in users and add the user data to the audits_by_user dictionary
for user_id, user_audits in audits_by_user.items():
    user = users[user_id]
    if user:
        audits_by_user[user_id]['user'] = {
            'id': user_id,
            'name': user.get('name'),
            'email': user.get('email'),
            'role': user.get('role')
        }
    else:
        audits_by_user[user_id]['user'] = {
            'id': user_id,
            'name': None,
            'email': None,
            'role': None
        } 
print(f'Added user data to audits_by_user, now contains {len(audits_by_user)} users with audits')

Added user data to audits_by_user, now contains 8 users with audits


In [63]:
user_id = '101'
user_audits = audits_by_user[user_id]
user_audits

{'5dab5f7d-e87f-4b58-b0b8-189c9ac3c1cb': 2,
 'total': 3759,
 'c66087a7-839b-45e6-a3d4-a474fe34bcb6': 20,
 '02e89794-a60e-4057-bfc6-17bd8b2abf97': 1,
 'c6fd8935-7cc1-48ce-ae12-9bf6348f81ee': 1,
 '6e5a867c-e2ae-438c-ae8a-ddba28b3a618': 1,
 'e29d1ab5-d21a-4328-aaa4-f86e82a12424': 2,
 '3f0074cf-e77a-44fe-9b60-05041adb4ff7': 17,
 '5c34a89d-664a-4566-bd77-6c7aca112086': 1,
 '967bc209-3b10-450a-848a-8f3a6b9503fd': 6,
 '2af23f8b-a4cd-4059-ac70-7243bdc023ed': 5,
 '0941d9e8-a13a-4a08-91ce-9b4f33087545': 4,
 'eb1e71f0-ecf8-4309-8fa0-ac96e533f031': 6,
 '6827b4f6-ab4f-441a-a2d8-70ebe7eeaf8f': 14,
 'd8ba3519-322a-46b2-8ab1-f2356dbf9edf': 3,
 'c7338aa6-badb-4fd8-bbfe-f0f1ff826873': 11,
 '6a212dc2-401d-4bfd-b6f5-f80a72a1e43e': 7,
 '28d5456b-1f23-4766-a348-f5a25fb4315a': 11,
 '0f6af519-76a7-4a94-b95e-5e66301ac81d': 12,
 'c128e84f-0707-4a05-bb0d-b13d05cb94dd': 3,
 '74e5a4b1-ca53-4380-babc-42dfa9c40004': 1,
 '1450443a-2788-4550-a545-c8780b35f6e0': 10,
 'e7b6e6b5-21ea-4e3a-a222-ef7164e20335': 12,
 '575b4b

In [71]:
# sort keys of the 101 user in audits_by_user by value
user_id = '101'
user_audits = audits_by_user[user_id]

def sort_user_audits(user_id):
    user_audits = audits_by_user[user_id]
    # sort the keys of the user_audits dictionary by value, some values are strings, some are ints, some are dicts
    # convert all values to ints
    user_audits = {key: int(value) if isinstance(value, str) else value for key, value in user_audits.items()}	
    # sort the keys by value
    return user_audits

sort_user_audits('101')


{'5dab5f7d-e87f-4b58-b0b8-189c9ac3c1cb': 2,
 'total': 3759,
 'c66087a7-839b-45e6-a3d4-a474fe34bcb6': 20,
 '02e89794-a60e-4057-bfc6-17bd8b2abf97': 1,
 'c6fd8935-7cc1-48ce-ae12-9bf6348f81ee': 1,
 '6e5a867c-e2ae-438c-ae8a-ddba28b3a618': 1,
 'e29d1ab5-d21a-4328-aaa4-f86e82a12424': 2,
 '3f0074cf-e77a-44fe-9b60-05041adb4ff7': 17,
 '5c34a89d-664a-4566-bd77-6c7aca112086': 1,
 '967bc209-3b10-450a-848a-8f3a6b9503fd': 6,
 '2af23f8b-a4cd-4059-ac70-7243bdc023ed': 5,
 '0941d9e8-a13a-4a08-91ce-9b4f33087545': 4,
 'eb1e71f0-ecf8-4309-8fa0-ac96e533f031': 6,
 '6827b4f6-ab4f-441a-a2d8-70ebe7eeaf8f': 14,
 'd8ba3519-322a-46b2-8ab1-f2356dbf9edf': 3,
 'c7338aa6-badb-4fd8-bbfe-f0f1ff826873': 11,
 '6a212dc2-401d-4bfd-b6f5-f80a72a1e43e': 7,
 '28d5456b-1f23-4766-a348-f5a25fb4315a': 11,
 '0f6af519-76a7-4a94-b95e-5e66301ac81d': 12,
 'c128e84f-0707-4a05-bb0d-b13d05cb94dd': 3,
 '74e5a4b1-ca53-4380-babc-42dfa9c40004': 1,
 '1450443a-2788-4550-a545-c8780b35f6e0': 10,
 'e7b6e6b5-21ea-4e3a-a222-ef7164e20335': 12,
 '575b4b

In [27]:
# calculate percentage for each user
total_audits = sum(user_data.get('total', 0) for user_data in audits_by_user.values())
for user_id, user_data in audits_by_user.items():
    user_data['percentage'] = (user_data.get('total', 0) / total_audits * 100) if total_audits > 0 else 0

for user in audits_by_user:
  print(f'User {user} {audits_by_user[user]["user"]["name"]} has {audits_by_user[user].get("total", 0)} audits ({audits_by_user[user]["percentage"]:.2f}%)')

User 101 Robert Shaw has 3759 audits (25.06%)
User 100 David Zbíral has 5224 audits (34.82%)
User 103 Katia Riccardo has 2045 audits (13.63%)
User 102 Davor Salihovic has 866 audits (5.77%)
User 02868c93-517e-4602-94b7-057ce1a393fd Katalin Suba has 2753 audits (18.35%)
User 1 admin has 202 audits (1.35%)
User 107 Larissa de Freitas Lyth has 145 audits (0.97%)
User 151 Tomáš Hampejs has 7 audits (0.05%)
